# Sala Chaturamuk Phaichit — 3D Gaussian Splatting (Kaggle)

Stage 2 of the reconstruction comparison. Stage 1 (COLMAP structure-from-motion) already ran locally and registered all 146 photographs of `salathai_version2` into a single model at 0.58 px mean reprojection error; this notebook consumes those camera poses. Nothing further runs on the laptop.

**This is a comparison, not a replacement.** The project's contribution is image-based rendering that synthesizes novel views *without* recovering geometry. This notebook answers the adjacent question: what do you get, and what does it cost, if you do recover geometry from the same photographs?

## Setup before running anything

1. **Upload the data once, as a Dataset.** Left sidebar → *Datasets* → *New Dataset* → upload `sala_v2_pinhole.zip` (105 MB). Name it `sala-v2-pinhole`. It then persists across every future session — unlike Colab, you never re-upload.
2. **Attach it:** right sidebar → *Input* → *Add Input* → your `sala-v2-pinhole` dataset.
3. **Turn the GPU on:** right sidebar → *Accelerator* → **GPU T4 ×2**.
4. **Turn the internet on:** right sidebar → *Internet* → **On**. Without this, `git clone` and `pip install` fail — it is the single most common reason this notebook stalls at step 3.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Right sidebar -> Accelerator -> GPU T4 x2.'

## 2. Unpack the data

`/kaggle/input/` is read-only, so the archive is expanded into `/kaggle/working/`, which is writable and is what gets saved as the notebook's output.

In [ ]:
import glob, os, zipfile, shutil

# 3DGS writes points3D.ply next to points3D.bin on first load, so the scene
# must live somewhere writable -- /kaggle/input is mounted read-only.
SCENE = '/kaggle/working/scene'

zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
if zips:
    print('extracting', zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('/kaggle/working/_extract')
    root = '/kaggle/working/_extract'
else:
    root = '/kaggle/input'

hits = glob.glob(f'{root}/**/sparse/0', recursive=True)
assert hits, f'No sparse/0 found under {root}. Is the dataset attached?'
src = os.path.dirname(os.path.dirname(hits[0]))

if os.path.abspath(src) != SCENE:
    shutil.rmtree(SCENE, ignore_errors=True)
    print('copying scene to writable storage...')
    shutil.copytree(src, SCENE)

n_images = len(glob.glob(f'{SCENE}/images/*.jpg'))
print('scene:', SCENE)
print('images:', n_images)
print('sparse:', sorted(os.listdir(f'{SCENE}/sparse/0')))
assert n_images > 0, f'No images under {SCENE}/images'


## 3. Install 3D Gaussian Splatting

The two submodules are CUDA extensions compiled against the runtime's toolkit, so this takes several minutes — the slowest setup step.

`TORCH_CUDA_ARCH_LIST` is set explicitly because the build otherwise probes the GPU and, on Kaggle, sometimes guesses wrong. `7.5` is the T4. Do **not** use the P100: it is compute capability 6.0, which Kaggle's PyTorch build no longer supports, so training aborts even though the extensions compile.

In [ ]:
import os
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'

%cd /kaggle/working
!git clone --recursive -q https://github.com/graphdeco-inria/gaussian-splatting
%cd /kaggle/working/gaussian-splatting
!pip install -q plyfile
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn

## 4. Train

30 000 iterations is the paper's default: roughly 30–50 minutes for 146 images. Kaggle guarantees the session far more reliably than Colab's free tier, but `--save_iterations` still writes intermediate models so an interruption never costs everything.

`--eval` holds out every 8th photograph for the measurement in the next cell.

In [ ]:
%cd /kaggle/working/gaussian-splatting
!python train.py \
  -s {SCENE} \
  -m /kaggle/working/output/sala_v2 \
  --iterations 30000 \
  --save_iterations 7000 15000 30000 \
  --eval


## 5. Measure it

Because training held out every 8th photograph, these PSNR/SSIM/LPIPS figures are scored against real photographs the model never saw.

This is the number that matters for the write-up: it is the *same kind of held-out measurement* the IBR pipeline's `src/evaluate.py` performs, so the two approaches can be compared on one footing rather than by eye.

In [ ]:
%cd /kaggle/working/gaussian-splatting
!python render.py -m /kaggle/working/output/sala_v2
!python metrics.py -m /kaggle/working/output/sala_v2

## 6. Collect the outputs

`point_cloud.ply` **is** the 3D model — a few hundred thousand oriented gaussians, each with position, covariance, opacity and view-dependent colour.

Everything left in `/kaggle/working/` is downloadable from the notebook's *Output* tab once it finishes. The clone and the unpacked images are deleted first so the output stays small enough to download comfortably.

To view it: drag the `.ply` onto <https://superspl.at/editor> and fly around it in the browser — the strongest way to show this live.

In [ ]:
import os, shutil

PLY = '/kaggle/working/output/sala_v2/point_cloud/iteration_30000/point_cloud.ply'
print('model exists:', os.path.isfile(PLY), '|',
      round(os.path.getsize(PLY) / 1e6, 1) if os.path.isfile(PLY) else 0, 'MB')

shutil.copy(PLY, '/kaggle/working/sala_v2_gaussians.ply')

# Held-out comparisons: renders/ is what the model produced, gt/ the real
# photograph. These are the paper figures.
shutil.make_archive('/kaggle/working/sala_v2_renders', 'zip',
                    '/kaggle/working/output/sala_v2/test')

# Drop the bulky intermediates so the Output tab stays quick to download.
shutil.rmtree('/kaggle/working/gaussian-splatting', ignore_errors=True)
shutil.rmtree('/kaggle/working/scene', ignore_errors=True)

print('\nready to download from the Output tab:')
for f in sorted(os.listdir('/kaggle/working')):
    p = f'/kaggle/working/{f}'
    if os.path.isfile(p):
        print(f'  {f}  ({os.path.getsize(p)/1e6:.1f} MB)')

## What to expect, honestly

Both captures are **single-height rings** — every photograph was taken from roughly eye level while walking around the pavilion. So:

- **The sides should reconstruct well.** Dense coverage, 2.4° mean angular spacing, good parallax.
- **The roof will not.** Nothing ever looked down on it, so expect noise or a hole above the eaves. That is a limitation of the capture, not of the method, and is worth reporting rather than cropping out of the demo.
- **Viewpoints far from the ring will degrade.** Gaussian splatting interpolates confidently near the training cameras and invents detail away from them.

The measured camera path (`recon/salathai_version2/camera_path.json`) puts the walk at **342°**, not a full revolution, ending 2.21 units from where it began where a normal step is 0.19 — so there is one wider gap in the ring, and the reconstruction should be weakest there.

## If the CUDA extensions refuse to build

This compiles nothing, and `splatfacto` is the same family of technique:

```bash
!pip install nerfstudio
!ns-train splatfacto --data /kaggle/working/sala_v2_colmap
```

`ns-train nerfacto` trains a NeRF instead — literally a neural network, and the more textbook answer if that phrasing matters for the assignment.